## Semantic Searching

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter,CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma


#utility
import numpy as np
from typing import List,Dict,Any
from sentence_transformers import SentenceTransformer


C:\Users\kanha\AppData\Local\Temp\ipykernel_23532\3752526940.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [3]:
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
import numpy as np

In [5]:
model=SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
text="""
Langchain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains,agents, moemory and retriever.
The Effel Tower is located in Paris.
France is a popular tourist destination.
"""

## Step 1: Split into sentence
sentences=[i.strip() for i in text.split("\n") if i.strip()]

## Step 2: Embed each sentence
embeddings=model.encode(sentences)

## Step 3: Initialize params
threshold=0.7
chunks=[]
current_chunk=[sentences[0]]

## Ste 4: Semantic grouping based on threshold
for i in range(1,len(sentences)):
    sim=cosine_similarity(
        [embeddings[i-1]],
        [embeddings[i]]
    )[0][0]
    if sim>threshold:
        current_chunk.append(sentences[1])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]
chunks.append(" ".join(current_chunk))
print("\n Semanric chunks")
for idx,chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n {chunk}")


 Semanric chunks

Chunk 1:
 Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
 You can create chains,agents, moemory and retriever.

Chunk 3:
 The Effel Tower is located in Paris.

Chunk 4:
 France is a popular tourist destination.


## RAG pipeline


In [7]:

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model


In [8]:
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_core.runnables import (RunnablePassthrough,RunnableMap)
from langchain_core.output_parsers import StrOutputParser
import os

In [9]:
class ThresholdChunker:
    def __init__(self,model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model=SentenceTransformer(model_name)
        self.threshold=threshold
    def split(self,text:str):
        sentences=[i.strip() for i in text.split("\n") if i.strip()]


        embeddings=model.encode(sentences)

        ## Step 3: Initialize params
        
        chunks=[]
        current_chunk=[sentences[0]]

        ## Ste 4: Semantic grouping based on threshold
        for i in range(1,len(sentences)):
            sim=cosine_similarity(
                [embeddings[i-1]],
                [embeddings[i]]
            )[0][0]
            if sim>self.threshold:
                current_chunk.append(sentences[1])
            else:
                chunks.append(" ".join(current_chunk))
                current_chunk=[sentences[i]]
        chunks.append(" ".join(current_chunk))
        return chunks
    def split_documents(self,docs):
        results=[]
        for doc in docs:
            for chunk in self.split(doc.page_content):
                results.append(Document(page_content=chunk,metadata=doc.metadata))
        return results

In [10]:
simple_text="""
Langchain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains,agents, moemory and retriever.
The Effel Tower is located in Paris.
France is a popular tourist destination."""
doc=Document(page_content=simple_text)
doc

Document(metadata={}, page_content='\nLangchain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains,agents, moemory and retriever.\nThe Effel Tower is located in Paris.\nFrance is a popular tourist destination.')

In [11]:
### CHunking
chunker=ThresholdChunker(threshold=0.7)
chunks=chunker.split_documents([doc])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [12]:
chunks

[Document(metadata={}, page_content='Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains,agents, moemory and retriever.'),
 Document(metadata={}, page_content='The Effel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [13]:
### Vectorstoe
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)

vs=FAISS.from_documents(chunks,embeddings)
retriever=vs.as_retriever()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
template="""
Ansert the question based on following context:
{context}
Question:{question}
"""
prompt=PromptTemplate.from_template(template)

In [ ]:
llm=init_chat_model(model="groq:llama-3.3-70b-versatile",api_key="API_KEY")
rag_chain=(
    RunnableMap(
        {
            "context":lambda x: retriever.invoke(x['question']),
            "question":lambda x:x['question'],
        }
    )|prompt|llm|StrOutputParser()
)
query={"question":"What is Langchain used for?"}
result=rag_chain.invoke(query)

In [16]:
print(result)

Langchain is a framework used for building applications with Large Language Models (LLMs). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone, allowing users to create chains, agents, memory, and retrievers.


## Semantic Chunker using Langchain

In [17]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader


In [18]:
loader=TextLoader("langchain_into.txt")
docs=loader.load()

## Initialize
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"

)
chunker=SemanticChunker(embeddings)
chunks=chunker.split_documents(docs)

for i ,chunk in enumerate(chunks):
    print(f"\n Chunks {i+1}....\n {chunk.page_content}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


 Chunks 1....
 Langchain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone. You can create chains,agents, moemory and retriever.

 Chunks 2....
 The Effel Tower is located in Paris. France is a popular tourist destination.
